### Classificação de grupo etário (young vs older)

Compara **5 modelos** em **3 tarefas** de representação das redes:

1. **EO** — métricas só em olhos abertos  
2. **EC** — métricas só em olhos fechados  
3. **Diff (EC − EO)** — contraste por participante  

Modelos: Dummy (baseline), Logistic Regression L2, SVM RBF, Random Forest, KNN após PCA.

- Alvo: `young` (`20–40`) vs `older` (`55–80`)  
- Unidade: um participante por linha em cada tarefa, então a CV estratificada não mistura EO e EC da mesma pessoa  
- Features: só métricas de rede (sem idade/gênero — evita vazamento)  
- CV: StratifiedKFold (5), métricas `accuracy`, `balanced_accuracy`, `roc_auc`  
- Amostra analítica: **54 sujeitos** (35 young / 19 older) após o **filtro saudável** do `build_dataset`  
  (metadados LEMON ~227 → intermediários de rede ~158 → processados aqui = filtrados)  
- Resultados **exploratórios** (p ≫ n); LogReg/SVM/RF usam `class_weight="balanced"`  
- Valores `média ± desvio` são a dispersão **entre folds**, não intervalos de confiança

**Features (215):** grau, clustering ponderado, intermediação e frequência de hub alinhados por eletrodo (53 eletrodos com cobertura ≥95% em EO e EC; `degree_Oz`, `betweenness_Cz`, …), mais 3 globais (`edges_mean`, `edges_std`, `edges_cv`). Notebook: `00_redes/03_realign_features_by_electrode.ipynb`.


### Imports


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

### Parâmetros e dados


In [2]:
DATASET_PATH = Path("../../data/processed/dataset_features.parquet")
RANDOM_STATE = 42
N_SPLITS = 5
PCA_COMPONENTS = 10
KNN_NEIGHBORS = 5

YOUNG_AGES = {"20-25", "25-30", "30-35", "35-40"}
OLDER_AGES = {"55-60", "60-65", "65-70", "70-75", "75-80"}

NODE_PREFIXES = (
    "degree_",
    "clustering_",
    "betweenness_",
    "hub_frequency_",
)
GLOBAL_FEATURES = [
    "edges_mean",
    "edges_std",
    "edges_cv",
]

df = pd.read_parquet(DATASET_PATH)
df = df[df["condition"].isin(["EO", "EC"])].copy()
df["age"] = df["age"].astype(str)

duplicates = df.duplicated(subset=["subject_id", "condition"], keep=False)
assert not duplicates.any(), (
    "Existem registros duplicados por participante e condição."
)


def map_age_group(age: str):
    if age in YOUNG_AGES:
        return "young"
    if age in OLDER_AGES:
        return "older"
    return None


df["age_group"] = df["age"].map(map_age_group)
df = df[df["age_group"].notna()].copy()
df["y"] = df["age_group"].map({"young": 0, "older": 1}).astype(int)

print(
    "Amostra analítica (filtro saudável): "
    f"{df['subject_id'].nunique()} sujeitos "
    f"(metadados LEMON ~227; intermediários ~158; processados = filtrados)."
)
print(df.drop_duplicates("subject_id")["age_group"].value_counts())
df[["subject_id", "condition", "age", "age_group", "y"]].head()

Amostra analítica (filtro saudável): 54 sujeitos (metadados LEMON ~227; intermediários ~158; processados = filtrados).
age_group
young    35
older    19
Name: count, dtype: int64


,subject_id,condition,age,age_group,y
0,sub-010069,EC,20-25,young,0
1,sub-010069,EO,20-25,young,0
2,sub-010070,EC,20-25,young,0
3,sub-010070,EO,20-25,young,0
4,sub-010079,EC,25-30,young,0


### Features de rede


In [3]:
FEATURE_COLS = [
    c
    for c in df.columns
    if c.startswith(NODE_PREFIXES) or c in GLOBAL_FEATURES
]
FEATURE_COLS = (
    df[FEATURE_COLS]
    .select_dtypes(include="number")
    .dropna(axis=1, how="all")
    .columns.tolist()
)
print(f"N features de rede: {len(FEATURE_COLS)}")

N features de rede: 215


### Matrizes por tarefa


In [4]:
def build_feature_matrix(frame: pd.DataFrame, condition: str):
    sub = frame[frame["condition"] == condition].copy()
    sub = sub.drop_duplicates("subject_id")
    X = sub[FEATURE_COLS].copy()
    y = sub["y"].copy()
    X.index = sub["subject_id"].values
    y.index = sub["subject_id"].values
    return X, y


def build_diff_matrix(frame: pd.DataFrame):
    eo = (
        frame[frame["condition"] == "EO"]
        .drop_duplicates("subject_id")
        .set_index("subject_id")
    )
    ec = (
        frame[frame["condition"] == "EC"]
        .drop_duplicates("subject_id")
        .set_index("subject_id")
    )
    common = eo.index.intersection(ec.index)
    X = ec.loc[common, FEATURE_COLS] - eo.loc[common, FEATURE_COLS]
    y = eo.loc[common, "y"]
    return X, y


X_eo, y_eo = build_feature_matrix(df, "EO")
X_ec, y_ec = build_feature_matrix(df, "EC")
X_diff, y_diff = build_diff_matrix(df)

tasks = {
    "EO": (X_eo, y_eo),
    "EC": (X_ec, y_ec),
    "Diff_EC_minus_EO": (X_diff, y_diff),
}

for name, (X, y) in tasks.items():
    print(f"{name}: X={X.shape}, y={y.value_counts().to_dict()}")

EO: X=(54, 215), y={0: 35, 1: 19}
EC: X=(54, 215), y={0: 35, 1: 19}
Diff_EC_minus_EO: X=(54, 215), y={0: 35, 1: 19}


### Modelos


In [5]:
def get_models():
    return {
        "Dummy": DummyClassifier(strategy="most_frequent"),
        "LogReg_L2": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                        max_iter=5000,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    )),
            ]
        ),
        "SVM_RBF": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "clf",
                    SVC(
                        kernel="rbf",
                        class_weight="balanced",
                        probability=True,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "RandomForest": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                (
                    "clf",
                    RandomForestClassifier(
                        n_estimators=300,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                ),
            ]
        ),
        "KNN_PCA": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=PCA_COMPONENTS, random_state=RANDOM_STATE)),
                ("clf", KNeighborsClassifier(n_neighbors=KNN_NEIGHBORS)),
            ]
        ),
    }


models = get_models()
list(models)

['Dummy', 'LogReg_L2', 'SVM_RBF', 'RandomForest', 'KNN_PCA']

### Avaliação (CV estratificada)


In [6]:
def evaluate_all(X, y, models, task: str) -> pd.DataFrame:
    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    scoring = {
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "roc_auc": "roc_auc",
    }
    rows = []
    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
        row = {
            "task": task,
            "model": name,
            "n": len(y),
            "accuracy_mean": float(np.mean(scores["test_accuracy"])),
            "accuracy_std": float(np.std(scores["test_accuracy"])),
            "balanced_accuracy_mean": float(np.mean(scores["test_balanced_accuracy"])),
            "balanced_accuracy_std": float(np.std(scores["test_balanced_accuracy"])),
            "roc_auc_mean": float(np.mean(scores["test_roc_auc"])),
            "roc_auc_std": float(np.std(scores["test_roc_auc"])),
        }
        rows.append(row)
        print(
            f"[{task}] {name}: "
            f"acc={row['accuracy_mean']:.3f}±{row['accuracy_std']:.3f} | "
            f"bal={row['balanced_accuracy_mean']:.3f}±{row['balanced_accuracy_std']:.3f} | "
            f"auc={row['roc_auc_mean']:.3f}±{row['roc_auc_std']:.3f}"
        )
    return pd.DataFrame(rows)


results = [evaluate_all(X, y, models, task) for task, (X, y) in tasks.items()]
df_results = pd.concat(results, ignore_index=True)
df_results

[EO] Dummy: acc=0.649±0.025 | bal=0.500±0.000 | auc=0.500±0.000
[EO] LogReg_L2: acc=0.720±0.107 | bal=0.675±0.144 | auc=0.714±0.171
[EO] SVM_RBF: acc=0.707±0.118 | bal=0.707±0.116 | auc=0.757±0.073
[EO] RandomForest: acc=0.669±0.087 | bal=0.627±0.123 | auc=0.762±0.144
[EO] KNN_PCA: acc=0.742±0.064 | bal=0.655±0.089 | auc=0.638±0.164
[EC] Dummy: acc=0.649±0.025 | bal=0.500±0.000 | auc=0.500±0.000
[EC] LogReg_L2: acc=0.616±0.173 | bal=0.593±0.212 | auc=0.652±0.197
[EC] SVM_RBF: acc=0.555±0.123 | bal=0.512±0.106 | auc=0.583±0.156
[EC] RandomForest: acc=0.649±0.063 | bal=0.562±0.059 | auc=0.661±0.201
[EC] KNN_PCA: acc=0.687±0.105 | bal=0.612±0.092 | auc=0.652±0.150
[Diff_EC_minus_EO] Dummy: acc=0.649±0.025 | bal=0.500±0.000 | auc=0.500±0.000
[Diff_EC_minus_EO] LogReg_L2: acc=0.662±0.143 | bal=0.589±0.176 | auc=0.562±0.243
[Diff_EC_minus_EO] SVM_RBF: acc=0.687±0.105 | bal=0.620±0.119 | auc=0.655±0.166
[Diff_EC_minus_EO] RandomForest: acc=0.685±0.091 | bal=0.620±0.093 | auc=0.643±0.090
[Diff

,task,model,n,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,roc_auc_mean,roc_auc_std
0,EO,Dummy,54,0.649091,0.025455,0.500000,0.000000,0.500000,0.000000
1,EO,LogReg_L2,54,0.720000,0.107010,0.675000,0.144367,0.714286,0.170533
2,EO,SVM_RBF,54,0.707273,0.117607,0.707143,0.115507,0.757143,0.072843
3,EO,RandomForest,54,0.669091,0.087121,0.627381,0.122602,0.761905,0.144239
4,EO,KNN_PCA,54,0.741818,0.064437,0.654762,0.089484,0.638095,0.164458
5,EC,Dummy,54,0.649091,0.025455,0.500000,0.000000,0.500000,0.000000
6,EC,LogReg_L2,54,0.616364,0.173291,0.592857,0.211711,0.652381,0.196540
7,EC,SVM_RBF,54,0.554545,0.123315,0.511905,0.105611,0.583333,0.156129
8,EC,RandomForest,54,0.649091,0.062879,0.561905,0.058877,0.660714,0.200764
9,EC,KNN_PCA,54,0.687273,0.105015,0.611905,0.092014,0.652381,0.150330


In [9]:
from pathlib import Path

MODEL_LABELS = {
    "Dummy": "Dummy",
    "LogReg_L2": "Regressão logística",
    "SVM_RBF": "SVM",
    "RandomForest": "Random Forest",
    "KNN_PCA": "KNN com PCA",
}
COLUMNS = {
    ("EO", "balanced_accuracy"): "Acurácia balanceada em EO",
    ("EO", "roc_auc"): "ROC-AUC em EO",
    ("EC", "roc_auc"): "ROC-AUC em EC",
    ("Diff_EC_minus_EO", "roc_auc"): "ROC-AUC em EC − EO",
}
BOLD_COLUMNS = ["Acurácia balanceada em EO", "ROC-AUC em EO"]


def fmt(mean: float, std: float) -> str:
    return f"{mean:.3f} ± {std:.3f}".replace(".", ",")


rows = {}
means = {}
for (task, metric), label in COLUMNS.items():
    sub = df_results[df_results["task"] == task].set_index("model")
    rows[label] = {
        MODEL_LABELS[m]: fmt(sub.loc[m, f"{metric}_mean"], sub.loc[m, f"{metric}_std"])
        for m in MODEL_LABELS
    }
    means[label] = {MODEL_LABELS[m]: sub.loc[m, f"{metric}_mean"] for m in MODEL_LABELS}

tabela4 = pd.DataFrame(rows).rename_axis("Modelo")
medias = pd.DataFrame(means)

for col in BOLD_COLUMNS:
    best = medias[col].idxmax()
    tabela4.loc[best, col] = f"<b>{tabela4.loc[best, col]}</b>"

styled = (
    tabela4.style
    .set_caption(
        "Tabela 4: Desempenho dos modelos na classificação dos grupos etários por condição "
        "de registro e pela diferença entre condições. Os valores em negrito destacam as maiores "
        "médias na condição EO, sem indicar superioridade estatisticamente demonstrada."
    )
    .set_table_styles([
        {"selector": "caption", "props": "caption-side: top; text-align: left; font-weight: bold;"},
        {"selector": "th, td", "props": "padding: 4px 10px; text-align: center; font-family: 'Times New Roman';"},
        {"selector": "th.row_heading", "props": "text-align: left;"},
    ])
)
display(styled)

out = Path("../../figures/tabela4_classificacao_idade")
out.with_suffix(".html").write_text(styled.to_html(), encoding="utf-8")
tabela4.replace(r"</?b>", "", regex=True).to_csv(out.with_suffix(".csv"), sep=";", encoding="utf-8-sig")
print(f"Salvo: {out.with_suffix('.html')} e {out.with_suffix('.csv')}")

,Acurácia balanceada em EO,ROC-AUC em EO,ROC-AUC em EC,ROC-AUC em EC − EO
Modelo,,,,
Dummy,"0,500 ± 0,000","0,500 ± 0,000","0,500 ± 0,000","0,500 ± 0,000"
Regressão logística,"0,675 ± 0,144","0,714 ± 0,171","0,652 ± 0,197","0,562 ± 0,243"
SVM,"0,707 ± 0,116","0,757 ± 0,073","0,583 ± 0,156","0,655 ± 0,166"
Random Forest,"0,627 ± 0,123","0,762 ± 0,144","0,661 ± 0,201","0,643 ± 0,090"
KNN com PCA,"0,655 ± 0,089","0,638 ± 0,164","0,652 ± 0,150","0,604 ± 0,101"


Salvo: ..\..\figures\tabela4_classificacao_idade.html e ..\..\figures\tabela4_classificacao_idade.csv


### Tabela resumo


In [7]:
summary = df_results.copy()
summary["accuracy"] = summary.apply(
    lambda r: f"{r['accuracy_mean']:.3f} ± {r['accuracy_std']:.3f}", axis=1
)
summary["balanced_accuracy"] = summary.apply(
    lambda r: f"{r['balanced_accuracy_mean']:.3f} ± {r['balanced_accuracy_std']:.3f}",
    axis=1,
)
summary["roc_auc"] = summary.apply(
    lambda r: f"{r['roc_auc_mean']:.3f} ± {r['roc_auc_std']:.3f}", axis=1
)
summary_pivot = summary.pivot(
    index="model", columns="task", values="balanced_accuracy"
)
summary_pivot

task,Diff_EC_minus_EO,EC,EO
model,,,
Dummy,0.500 ± 0.000,0.500 ± 0.000,0.500 ± 0.000
KNN_PCA,0.587 ± 0.107,0.612 ± 0.092,0.655 ± 0.089
LogReg_L2,0.589 ± 0.176,0.593 ± 0.212,0.675 ± 0.144
RandomForest,0.620 ± 0.093,0.562 ± 0.059,0.627 ± 0.123
SVM_RBF,0.620 ± 0.119,0.512 ± 0.106,0.707 ± 0.116


In [8]:
fig = px.bar(
    df_results,
    x="model",
    y="balanced_accuracy_mean",
    error_y="balanced_accuracy_std",
    color="task",
    barmode="group",
    title="Balanced accuracy por modelo e tarefa (CV 5-fold)",
    labels={
        "balanced_accuracy_mean": "Balanced accuracy",
        "model": "Modelo",
        "task": "Tarefa",
    },
)
fig.update_layout(yaxis_range=[0, 1], template="plotly_white", height=450)
fig.show()

### Interpretação (guia)

- **Dummy (`most_frequent`)** prediz sempre a classe majoritária (`young` ≈ 65% dos sujeitos). Accuracy alta no Dummy **não** indica sinal; use **balanced accuracy** e **ROC-AUC**.
- Comparar cada modelo ao Dummy na mesma tarefa: só há evidência útil se balanced accuracy / AUC forem consistentemente superiores.
- **EO vs EC vs Diff**: se Diff for melhor, o contraste de estado carrega informação etária além do nível absoluto; se EO ou EC forem melhores, o estado isolado já basta.
- `média ± desvio` = dispersão **entre os 5 folds** (não é intervalo de confiança).
- Funil da amostra: metadados (~227) → intermediários de rede (~158) → processados após filtro saudável (**54**). Ampliar a amostra (ou revisitar critérios de inclusão) é o passo mais relevante para confiabilidade.
- n = 54 e CV 5-fold → resultados exploratórios (sem grid search).

Próximo passo (permutação EO, LogReg/SVM): `08_ml_age_permutation_eo.ipynb`.

Robustez / interpretabilidade (EO + SVM): `09_ml_age_robustness_interpretability.ipynb`.


### Amostra

Funil da amostra: metadados (227) → participantes com redes processadas (158) → amostra analítica após os critérios de inclusão saudável (54). A documentação dos critérios de inclusão e a avaliação de sua influência sobre a composição etária da amostra são necessárias para interpretar a capacidade de generalização dos resultados.

Na amostra analítica de 54 participantes, os modelos treinados com métricas das redes na condição de olhos abertos apresentaram médias superiores às do classificador de referência. O SVM teve o desempenho mais equilibrado em EO (acurácia balanceada de 0,707 ± 0,116; ROC-AUC de 0,757 ± 0,073). A Regressão Logística alcançou acurácia balanceada de 0,675 ± 0,144 e ROC-AUC de 0,714 ± 0,171, e o Random Forest, acurácia balanceada de 0,627 ± 0,123 e ROC-AUC de 0,762 ± 0,144, com maior variação entre os folds. Os resultados obtidos na condição de olhos fechados foram mais modestos (ROC-AUC entre 0,58 e 0,66) e apresentaram maior variabilidade entre os folds. As diferenças entre as métricas de EC e EO não demonstraram capacidade discriminativa clara (ROC-AUC entre 0,56 e 0,66). Esses achados sugerem, de forma exploratória, que as redes na condição de olhos abertos podem conter informações associadas ao grupo etário. Entretanto, o número reduzido de participantes e a elevada quantidade de características exigem validação adicional.